In [0]:
SELECT current_catalog();

### 1. Create a database

In [0]:
create database if not exists demo;

In [0]:
show databases;

In [0]:
describe database extended demo;

In [0]:
select current_database();

In [0]:
show tables;

In [0]:
show tables in demo;

In [0]:
use demo;

In [0]:
select current_database()

### 2.1 Create MANAGED TABLE (python)

In [0]:
%run "../utils/config"

In [0]:
%python
race_restults_df = spark.read.format("parquet").load(f"{presentation_folder_path}/race_results")

In [0]:
%python
race_restults_df.write.mode("overwrite").saveAsTable("demo.race_results_python")

In [0]:
select * from demo.race_results_python
where race_year = 2020

In [0]:
describe table extended demo.race_results_python

### 2.2 Create MANAGE TABLE (sql)

In [0]:
CREATE TABLE demo.race_results_sql AS
select * from demo.race_results_python
where race_year = 2020


In [0]:
describe table extended demo.race_results_sql

In [0]:
drop table demo.race_results_sql -- this will also drop the files because is a managed table 

In [0]:
SHOW TABLES IN demo

### 3.1 Create EXTERNAL TABLE (python)

In [0]:
%python
demo_table_name_py = "race_results_ext_py"

In [0]:
%python
race_restults_df.write.mode("overwrite").option("path",f"{presentation_folder_path}/{demo_table_name_py}" ).saveAsTable(f"{demo_table_name_py}")

In [0]:
describe table extended demo.race_results_ext_py

### 3.2 Create EXTERNAL TABLE (sql)

In [0]:
%python
demo_table_name_sql = "race_results_ext_sql"

In [0]:
%python
spark.sql(
f"""
CREATE TABLE IF NOT EXISTS demo.{demo_table_name_sql}
(
  race_year INT,
  race_name STRING,
  race_date TIMESTAMP,
  circuit_location STRING,
  driver_name STRING,
  driver_number INT,
  driver_nationality STRING,
  team_nationality STRING,
  team STRING,
  grid INT,
  fastest_lap INT,
  race_time STRING,
  points FLOAT,
  position INT,
  ingestion_date TIMESTAMP
)
USING parquet
LOCATION "{presentation_folder_path}/{demo_table_name_sql}"
""")


In [0]:
describe table demo.race_results_ext_py;

In [0]:
insert into demo.race_results_ext_sql --order of columns matter 
select race_year, race_name, race_date, circuit_location,
       driver_name, driver_number, driver_nationality,
       team_nationality, team, grid, fastest_lap,
       race_time, points, position, ingestion_date
from demo.race_results_ext_py
where race_year = 2020

In [0]:
describe table extended demo.race_results_ext_sql

In [0]:
drop table demo.race_results_ext_py;
drop table demo.race_results_ext_sql;

### 4.1 Create Temp Views

In [0]:
CREATE OR REPLACE TEMP VIEW t_race_results AS 
SELECT * FROM demo.race_results_python
WHERE race_year = 2020

In [0]:
select * from t_race_results

### 4.2 Create Gloval Temp View

In [0]:
CREATE OR REPLACE GLOBAL TEMP VIEW g_race_results AS 
select * from demo.race_results_python
where race_year = 2019

### 4.3 Create View

In [0]:
CREATE OR REPLACE VIEW demo.p_race_results AS 
select * from demo.race_results_python
where race_year = 2022